# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [3]:
import os
os.getcwd()

'/home/fernando/projects/llm_engineering/week1/community-contributions/1_thecirocks'

In [4]:
import os
os.listdir('.')

['thecirocks_week1_day5.ipynb', 'thecirocks_week1_day2.ipynb']

In [5]:
!find ~/projects/llm_engineering -name "scraper.py" 2>/dev/null

/home/fernando/projects/llm_engineering/week2/scraper.py
/home/fernando/projects/llm_engineering/week2/community-contributions/Joe-Mwangi/scraper.py
/home/fernando/projects/llm_engineering/week2/community-contributions/Shield01/scraper.py
/home/fernando/projects/llm_engineering/week2/community-contributions/temmyogunbo/scraper.py
/home/fernando/projects/llm_engineering/week2/community-contributions/job-one-pager-gradio/scraper.py
/home/fernando/projects/llm_engineering/week5/community-contributions/haastrupea/rag_pipeline/scraper.py
/home/fernando/projects/llm_engineering/week3/community-contributions/job-description-one-pager-gradio/scraper.py
/home/fernando/projects/llm_engineering/week1/scraper.py
/home/fernando/projects/llm_engineering/week1/community-contributions/Omtosho Joseph/scraper.py
/home/fernando/projects/llm_engineering/week1/community-contributions/anadi_sharma_15/scraper.py
/home/fernando/projects/llm_engineering/week1/community-contributions/contributor_wk/scraper.py
/

In [6]:
import shutil
shutil.copy('/home/fernando/projects/llm_engineering/week1/scraper.py', '.')

'./scraper.py'

In [7]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [8]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [9]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [10]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [11]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [12]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [17]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [18]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [19]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [20]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 6 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company/product page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter/X profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [21]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'models page', 'url': 'https://huggingface.co/models'},
  {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'Discourse community', 'url': 'https://discuss.huggingface.co'},
  {'type': 'Status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'Endpoints page', 'url': 'https://endpoints.huggingf

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [22]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [23]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4-Pro
Updated
3 days ago
•
174k
•
3.24k
deepseek-ai/DeepSeek-V4-Flash
Updated
3 days ago
•
96.9k
•
857
openai/privacy-filter
Updated
7 days ago
•
57.7k
•
1.09k
Qwen/Qwen3.6-27B
Updated
6 days ago
•
509k
•
1.01k
moonshotai/Kimi-K2.6
Updated
about 22 hours ago
•
489k
•
1.15k
Browse 2M+ models
Spaces
Running
on
CPU Upgrade
259
ML Intern
🤖
259
Explore ML concepts and get help with a virtual intern
Running
on
Zero
MCP
2.46k
Wan2.2 14B Preview
🐌
2.46k
generate a video from an image with a text prompt
Running
on
Zero

In [24]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [25]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [26]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ndeepseek-ai/DeepSeek-V4-Pro\nUpdated\n3 days ago\n•\n174k\n•\n3.24k\ndeepseek-ai/DeepSeek-V4-Flash\nUpdated\n3 days ago\n•\n96.9k\n•\n857\nopenai/privacy-filter\nUpdated\n7 days ago\n•\n57.7k\n•\n1.09k\nQwen/Qwen3.6-27B\nUpdated\n6 days ago\n•\n509k\n•\n1.01k\nmoonshotai/Kimi-K2.6\nUpdated\nabout 22 hours ago\n•\n489k\n•\n1.15k\nBrowse 2M+ models\nSpaces\nRunning\non\nCPU Upgrade\n259\nM

In [27]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [28]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the AI community building the future. It is the leading platform where the global machine learning (ML) community collaborates on models, datasets, and applications across all modalities including text, image, video, audio, and 3D. With over 2 million models and 500,000 datasets available, Hugging Face empowers creators, researchers, and enterprises to create, discover, and advance AI technologies collaboratively.

---

## Platform Highlights

- **Models:** Access and contribute to over 2 million ML models covering a broad range of applications.
- **Datasets:** Explore and build with more than 500,000 datasets updated regularly for AI training and evaluation.
- **Spaces:** Host and run ML-powered applications easily with scalable compute, including CPU and ZeroGPU options.
- **Collaboration:** Share and showcase your ML projects to build your professional AI portfolio while engaging with a vibrant community.
- **Open Source Stack:** Drive faster innovation with free and open-source tools that support a diverse range of AI modalities and implementations.

---

## Enterprise Solutions

Designed for organizations scaling their AI initiatives, Hugging Face offers:

- **Team Plans** starting at $20/user/month for instant team setup.
- **Enterprise Plans** with tailored contracts, enterprise-grade security, and dedicated premium support.
- **Security & Compliance:** Features like Single Sign-On (SSO), region selection and audit logs for data governance.
- **Granular Access Control:** Manage resource groups, token permissions, and repository visibility with fine control.
- **Advanced Compute:** Enhanced scalability with premium compute options, including ZeroGPU quota boosts.
- **Private Storage & Dataset Access:** Additional storage per member and private dataset viewing for secure collaboration.
- **Billing & Analytics:** Centralized billing with usage analytics and budget control.
- **Priority Support:** Direct access to expert Hugging Face support for seamless platform utilization.

---

## Pricing Overview

- **PRO Account:** $9/month for individual users offering boosted storage, inference credits, highest queue priority, and special features such as personal blog publishing.
- **Team Plan:** $20/user/month for teams to unify workflows and scale AI development collaboratively.

---

## Company Culture

Hugging Face champions an open, collaborative, and inclusive culture centered around community-driven innovation. The company actively fosters engagement by empowering users from hobbyists to AI experts to build, share, and refine AI technologies together. Transparency and accessibility are core values emphasized by their open-source ethos and vibrant online ecosystem.

---

## Careers & Opportunities

Hugging Face welcomes talent passionate about advancing AI technology and values contributors who thrive in a dynamic, community-oriented environment. Employees and collaborators benefit from access to cutting-edge AI tools, close interaction with the global AI community, and the opportunity to impact the future of machine learning.

Potential candidates can expect to work at the forefront of AI research and development, joining a mission-driven team that supports continuous learning, innovation, and meaningful contributions to open source.

---

## Why Choose Hugging Face?

- **Leading AI Community:** Largest collaborative platform with millions of models and datasets.
- **Versatile & Scalable:** Supports all data modalities and adapts from individual users to large enterprises.
- **Enterprise Ready:** Robust security, management, and support designed for business needs.
- **Open Source Commitment:** Building AI in the open ensures innovation is shared and accelerated globally.
- **Strong Support & Resources:** Comprehensive documentation, active community, and priority customer support.

---

## Connect with Hugging Face

- Explore AI apps and ML models: [huggingface.co](https://huggingface.co)
- Join the community and start collaborating today.
- Learn more about enterprise solutions and pricing on their website.

---

Build the future of AI with Hugging Face—where innovation meets collaboration.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [31]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [32]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future. It is the premier platform where the machine learning (ML) community collaborates by creating, discovering, and sharing ML models, datasets, and applications. Catering to researchers, developers, and enterprises worldwide, Hugging Face fosters open collaboration on cutting-edge AI technology to accelerate innovation in machine learning across various modalities including text, image, video, audio, and even 3D.

---

## Platform & Offerings

- **Massive Model Repository:** Access over 2 million open-source ML models covering tasks from natural language processing to computer vision.
- **Extensive Dataset Library:** Browse through 500,000+ datasets curated and continuously updated for ML training and research.
- **Spaces & Applications:** Experiment with 1 million+ AI applications and interactive demos hosted directly on the platform.
- **Collaboration Tools:** Host unlimited public models, datasets, and applications to collaborate seamlessly with the global AI community.
- **Open Source Stack:** Benefit from an open-source infrastructure that accelerates ML development across all modalities.
- **Portfolio Building:** Share your ML projects publicly and build a professional profile recognized by peers and potential employers.

---

## Enterprise Solutions

Hugging Face offers dedicated Team and Enterprise plans designed for organizations needing scalable, secure, and flexible AI platforms:

- **Team Plan:** Starting at $20/user/month, ideal for collaborative ML development.
- **Enterprise Plan:** Custom contracts with advanced features including:
  - Enterprise-grade security and access controls
  - Single Sign-On (SSO) for secure identity management
  - Region selection for repository data storage and auditability
  - Comprehensive audit logs to monitor actions across teams
  - Dedicated support and service SLAs

These offerings empower businesses to integrate AI safely and efficiently while maintaining regulatory compliance.

---

## Company Culture & Community

Hugging Face thrives on a vibrant community-driven culture that emphasizes:

- **Open Collaboration:** Encourages sharing, transparency, and collective progress in AI research and deployment.
- **Innovation at Scale:** Supports creators and enterprises alike to push the boundaries of AI applications across many domains.
- **Inclusivity:** Welcomes individuals from all backgrounds to contribute, learn, and grow in the field of machine learning.
- **Continuous Learning:** Offers resources, tutorials, and virtual tools like an ML Intern to enhance understanding and hands-on experience in AI concepts.

As a community-first company, Hugging Face values curiosity, teamwork, and the democratization of AI technologies.

---

## Customers & Impact

Hugging Face serves millions of ML practitioners, researchers, and enterprises worldwide, including leading AI research groups, technology startups, and global organizations. Its platform enables rapid prototyping, robust model deployment, and scalable ML solutions critical for advancing AI capabilities in real-world applications like privacy filters, image editing, video generation, and high-fidelity 3D content creation.

---

## Careers & Opportunities

Hugging Face is continuously growing and invites talented individuals passionate about AI to join its mission. Careers at Hugging Face include roles in:

- Machine Learning Research & Engineering
- Software Development and Infrastructure
- Product Management and Design
- Customer Success and Enterprise Support
- Community Engagement and Developer Advocacy

Employees benefit from being part of a forward-thinking technology company that values creativity, collaboration, and impact in shaping the future of AI.

---

## Get Started

Experience the power of collaborative AI development today:

- Explore AI applications and models: [Hugging Face Platform](https://huggingface.co)
- Sign up for free to create and share your own ML projects
- Contact sales for enterprise solutions and team subscriptions

Join Hugging Face — where the machine learning community builds the future together.

---

**Hugging Face**  
*The Home of Machine Learning*  
Website: https://huggingface.co  
Contact: sales@huggingface.co

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>